# Internship Task: Top 10 Products by Sales

**Objective:** Identify the 10 products with the highest total sales using the Superstore CSV dataset.

**Tools:** Python, pandas, matplotlib

Keep your Superstore CSV in the same folder as this notebook. If the filename differs, update `CSV_PATH` in the loading cell.

## Cell 1 — Import libraries

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

print('Libraries imported successfully.')

## Cell 2 — Locate and load the Superstore CSV

In [ ]:
# Change this if your CSV has a different filename or location.
# Example: CSV_PATH = Path('Sample - Superstore.csv')
CSV_PATH = None

if CSV_PATH is not None:
    csv_file = Path(CSV_PATH)
    if not csv_file.exists():
        raise FileNotFoundError(f'CSV file not found: {csv_file}')
else:
    csv_files = sorted(Path.cwd().glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(
            'No CSV found in this folder. Place your Superstore CSV here, '
            'or set CSV_PATH to its path.'
        )
    preferred = [p for p in csv_files if 'superstore' in p.name.lower() or 'sample' in p.name.lower()]
    csv_file = preferred[0] if preferred else csv_files[0]

df = pd.read_csv(csv_file, encoding='utf-8-sig')
print(f'Loaded file: {csv_file.resolve()}')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

## Cell 3 — Inspect columns and identify Product and Sales

In [ ]:
print('Columns in the dataset:')
print(df.columns.tolist())

def find_column(columns, candidates):
    normalized = {str(col).strip().lower().replace('_', ' '): col for col in columns}
    for candidate in candidates:
        key = candidate.strip().lower().replace('_', ' ')
        if key in normalized:
            return normalized[key]
    return None

product_col = find_column(df.columns, ['Product Name', 'Product', 'ProductName', 'Item Name', 'Item'])
sales_col = find_column(df.columns, ['Sales', 'Total Sales', 'Revenue', 'Amount'])

if product_col is None:
    raise KeyError("Product column not detected. Set product_col manually using a column name printed above.")
if sales_col is None:
    raise KeyError("Sales column not detected. Set sales_col manually using a column name printed above.")

print(f'Product column: {product_col}')
print(f'Sales column: {sales_col}')

## Cell 4 — Clean the Product and Sales fields

In [ ]:
work = df[[product_col, sales_col]].copy()
work[product_col] = work[product_col].astype('string').str.strip()
work[sales_col] = pd.to_numeric(
    work[sales_col].astype(str).str.replace(r'[$,]', '', regex=True),
    errors='coerce'
)

rows_before = len(work)
work = work.dropna(subset=[product_col, sales_col])
work = work[work[product_col] != '']

print(f'Valid rows: {len(work):,}')
print(f'Rows excluded: {rows_before - len(work):,}')
display(work.head())

## Cell 5 — Calculate total sales for each product and rank them

In [ ]:
product_sales = (
    work.groupby(product_col, as_index=False)[sales_col]
    .sum()
    .rename(columns={product_col: 'Product', sales_col: 'Total Sales'})
    .sort_values('Total Sales', ascending=False, kind='stable')
    .reset_index(drop=True)
)

top10 = product_sales.head(10).copy()
top10.insert(0, 'Rank', range(1, len(top10) + 1))

display(top10.style.format({'Total Sales': '{:,.2f}'}))

## Cell 6 — Check for ties at the 10th-place cutoff

In [ ]:
if len(product_sales) >= 10:
    cutoff_sales = product_sales.iloc[9]['Total Sales']
    tied_products = product_sales[
        product_sales['Total Sales'].round(10) == round(float(cutoff_sales), 10)
    ]
    print(f'10th-place sales cutoff: {cutoff_sales:,.2f}')
    if len(tied_products) > 1:
        print('Tie detected: these products share the cutoff sales value.')
        display(tied_products)
    else:
        print('No tie at the 10th-place cutoff.')
else:
    print(f'Only {len(product_sales)} distinct products were found.')

## Cell 7 — Visualize the Top 10 products

In [ ]:
chart_data = top10.sort_values('Total Sales', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(chart_data['Product'], chart_data['Total Sales'])
ax.set_title('Top 10 Products by Total Sales', fontsize=16, pad=14)
ax.set_xlabel('Total Sales')
ax.set_ylabel('Product')
ax.grid(axis='x', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.bar_label(bars, labels=[f'{value:,.0f}' for value in chart_data['Total Sales']], padding=4)
plt.tight_layout()
plt.show()

## Cell 8 — Export the Top 10 table

In [ ]:
output_file = Path.cwd() / 'top_10_products_by_sales.csv'
top10.to_csv(output_file, index=False)
print(f'Results saved to: {output_file.resolve()}')

## Cell 9 — Print a short summary for your report

In [ ]:
if not top10.empty:
    best_product = top10.iloc[0]
    print('TOP 10 PRODUCTS BY SALES — SUMMARY')
    print(f"Distinct products analyzed: {product_sales['Product'].nunique():,}")
    print(f"Highest-selling product: {best_product['Product']}")
    print(f"Its total sales: {best_product['Total Sales']:,.2f}")
    print(f'Products in the final table: {len(top10)}')
    print('Method: grouped rows by product, summed Sales, sorted descending, and selected the first 10 products.')

## Interview notes

**Why use Top 10 analysis?** It identifies high-revenue products and helps teams prioritize inventory, promotions, and further investigation.

**How are ties handled?** This notebook reports ties at the 10th-place sales cutoff. The displayed table still contains at most 10 rows.

**Measure used:** This task ranks by total sales revenue, not quantity sold or profit.